# Ablation study — trains YOLO26m-det on the caries dataset without external (Chaudhary) negative images, and compares false-positive rates against the full model. Supports the discussion in Section 4 on the effect of external negatives.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
import time
import glob

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch

# Ensure ultralytics is installed before importing
!pip install ultralytics

from ultralytics import YOLO

print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna — revisa el entorno de ejecución")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA disponible: False
GPU: ninguna — revisa el entorno de ejecución


## CELL 1 — Ablation dataset: only Ahmed's negatives (excluding Chaudhary)

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES")
ORIGEN_SIN_EXTERNOS = BASE / "Dataset Tratado 2"          # salida de Tratado_2, ya existe
DESTINO_ABLACION = BASE / "Dataset-Caries-SinNegativosExternos-Ablacion"  # copia para el experimento

assert ORIGEN_SIN_EXTERNOS.exists(), "Corre primero Tratado_2_dataset_Ahmed.md"

if DESTINO_ABLACION.exists():
    shutil.rmtree(DESTINO_ABLACION)
shutil.copytree(ORIGEN_SIN_EXTERNOS, DESTINO_ABLACION)

print("Dataset de ablación (sin negativos externos) listo en:", DESTINO_ABLACION)
for split in ["train", "val", "test"]:
    imgs = list((DESTINO_ABLACION / "images" / split).glob("*"))
    n_neg = sum(
        1 for img_p in imgs
        if (DESTINO_ABLACION / "labels" / split / f"{img_p.stem}.txt").exists()
        and not open(DESTINO_ABLACION / "labels" / split / f"{img_p.stem}.txt").read().strip()
    )
    print(f"  {split}: {len(imgs)} imágenes, {n_neg} negativas")


Dataset de ablación (sin negativos externos) listo en: /content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES/Dataset-Caries-SinNegativosExternos-Ablacion
  train: 1028 imágenes, 31 negativas
  val: 211 imágenes, 3 negativas
  test: 206 imágenes, 2 negativas


## Cell 2 — Train YOLO26m-det on the dataset without external negative examples

In [ ]:
RUNS_DRIVE = BASE / "runs_deteccion_caries"
RUNS_DRIVE.mkdir(parents=True, exist_ok=True)

EPOCHS = 150
PATIENCE = 30
IMGSZ = 640
OPTIMIZER = "AdamW"
LR0 = 0.001
LRF = 0.01
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
SEED = 42

print(f"yolo26m-det (sin negativos externos): pesos=yolo26m.pt, batch=8, epochs={EPOCHS}, "
      f"imgsz={IMGSZ}, patience={PATIENCE}, lr0={LR0}, pretrained=True")

t0 = time.time()

modelo = YOLO("yolo26m.pt")
resultado = modelo.train(
    data=str(DESTINO_ABLACION / "data.yaml"),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=8,
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    cos_lr=True,
    seed=SEED,
    pretrained=True,
    device=0,
    project=str(RUNS_DRIVE),
    name="yolo26m_det_sin_negativos_externos",
    exist_ok=True,
    plots=True,
    verbose=True,
)

tiempo_entrenamiento = time.time() - t0
save_dir = Path(resultado.save_dir)
resultado_ablacion = {
    "modelo": modelo,
    "resultado_train": resultado,
    "tiempo_entrenamiento_s": tiempo_entrenamiento,
    "save_dir": save_dir,
    "ruta_pesos_best": save_dir / "weights" / "best.pt",
    "ruta_pesos_last": save_dir / "weights" / "last.pt",
}
print(f"\nyolo26m_det_sin_negativos_externos entrenado en {tiempo_entrenamiento/60:.1f} min. "
      f"Pesos guardados en: {resultado_ablacion['ruta_pesos_best']}")

yolo26m-det (sin negativos externos): pesos=yolo26m.pt, batch=8, epochs=150, imgsz=640, patience=30, lr0=0.001, pretrained=True
Ultralytics 8.4.114 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES/Dataset-Caries-SinNegativosExternos/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, io

### CELL 2B — Save the ablation model

In [ ]:
import shutil
from pathlib import Path

MODELOS_DRIVE = BASE.parent / "MODELO CARIES"
MODELOS_DRIVE.mkdir(parents=True, exist_ok=True)

RUTA_DESTINO_BEST = MODELOS_DRIVE / "yolo26m_det_sin_negativos_externos_best.pt"
RUTA_DESTINO_LAST = MODELOS_DRIVE / "yolo26m_det_sin_negativos_externos_last.pt"

shutil.copy2(resultado_ablacion["ruta_pesos_best"], RUTA_DESTINO_BEST)
shutil.copy2(resultado_ablacion["ruta_pesos_last"], RUTA_DESTINO_LAST)

assert RUTA_DESTINO_BEST.exists(), f"No se copió {RUTA_DESTINO_BEST}"
assert RUTA_DESTINO_LAST.exists(), f"No se copió {RUTA_DESTINO_LAST}"

print("Modelo de ablación guardado en:")
print(" best:", RUTA_DESTINO_BEST)
print(" last:", RUTA_DESTINO_LAST)

## CELL 3 — Comparative evaluation using the same test set (v2, with external negatives)

In [ ]:
DATASET_V2 = BASE / "Dataset Tratado 3"

modelo_con_externos = YOLO(str("/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/MODELO CARIES/yolo26m-det_best.pt"))
modelo_sin_externos = YOLO(str("/content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/MODELO CARIES/yolo26m_det_sin_negativos_externos_best.pt"))

metrics_con = modelo_con_externos.val(data=str(DATASET_V2 / "data.yaml"), split="test")
metrics_sin = modelo_sin_externos.val(data=str(DATASET_V2 / "data.yaml"), split="test")

print("--- CON negativos externos (Chaudhary) ---")
print(f"mAP50={metrics_con.box.map50:.4f}  precisión={metrics_con.box.mp:.4f}  recall={metrics_con.box.mr:.4f}")
print("--- SIN negativos externos (solo Ahmed) ---")
print(f"mAP50={metrics_sin.box.map50:.4f}  precisión={metrics_sin.box.mp:.4f}  recall={metrics_sin.box.mr:.4f}")


Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 68.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 2.1±2.3 ms, read: 8.9±17.7 MB/s, size: 397.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/UNIVERSIDAD/9NO SEMESTRE/TITULACION/DATASET CARIES/Dataset Tratado 3/labels/test.cache... 242 images, 38 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 242/242 44.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 26.2s/it 6:59
                   all        242        693      0.843      0.915      0.927      0.721
Speed: 4.5ms preprocess, 1696.4ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/runs/detect/val
Ultralytics 8.4.115 🚀 Python-3.12.13 torch

## Diagnostic cells — reconciling negative-image counts (not required to reproduce the ablation result).

### CELL 4 — FP Rate by Source of Negative Result, Both Models

In [ ]:
def tasa_fp(modelo, carpeta_test, incluir_prefijo=None, excluir_prefijo=None):
    imgs_neg = []
    for img_p in Path(carpeta_test).glob("*"):
        lbl_p = Path(str(img_p).replace("images", "labels")).with_suffix(".txt")
        if not (lbl_p.exists() and not lbl_p.read_text().strip()):
            continue
        if incluir_prefijo is not None and not img_p.name.startswith(incluir_prefijo):
            continue
        if excluir_prefijo is not None and img_p.name.startswith(excluir_prefijo):
            continue
        imgs_neg.append(img_p)
    if not imgs_neg:
        return None, 0
    n_fp = 0
    for img_p in imgs_neg:
        res = modelo.predict(source=str(img_p), conf=0.25, verbose=False)[0]
        if res.boxes is not None and len(res.boxes) > 0:
            n_fp += 1
    return n_fp / len(imgs_neg), len(imgs_neg)

carpeta_test_v2 = DATASET_V2 / "images" / "test"

for nombre_modelo, modelo in [("CON externos", modelo_con_externos), ("SIN externos", modelo_sin_externos)]:
    # Ahmed genuino: NO tiene el prefijo healthy_
    fp_ahmed, n_ahmed = tasa_fp(modelo, carpeta_test_v2, excluir_prefijo="healthy_")
    # Chaudhary: SÍ tiene el prefijo healthy_
    fp_chaudhary, n_chaudhary = tasa_fp(modelo, carpeta_test_v2, incluir_prefijo="healthy_")
    print(f"\n--- {nombre_modelo} ---")
    print(f"  Negativos Ahmed (genuinos, sin prefijo healthy_): n={n_ahmed}  FP%={fp_ahmed:.1%}" if fp_ahmed is not None else "  Sin negativos Ahmed genuinos en test")
    print(f"  Negativos Chaudhary (healthy_):                    n={n_chaudhary}  FP%={fp_chaudhary:.1%}" if fp_chaudhary is not None else "  Sin negativos Chaudhary en test")



--- CON externos ---
  Negativos Ahmed (genuinos, sin prefijo healthy_): n=2  FP%=0.0%
  Negativos Chaudhary (healthy_):                    n=36  FP%=2.8%

--- SIN externos ---
  Negativos Ahmed (genuinos, sin prefijo healthy_): n=2  FP%=0.0%
  Negativos Chaudhary (healthy_):                    n=36  FP%=11.1%


### CELL 4B — FP rate by source of the negative result, based on the ENTIRE v2 dataset (not just the test set)


In [ ]:
def tasa_fp_dataset_completo(modelo, dataset_root, incluir_prefijo=None, excluir_prefijo=None):
    imgs_neg = []
    for split in ["train", "val", "test"]:
        carpeta = Path(dataset_root) / "images" / split
        for img_p in carpeta.glob("*"):
            lbl_p = Path(str(img_p).replace("images", "labels")).with_suffix(".txt")
            if not (lbl_p.exists() and not lbl_p.read_text().strip()):
                continue
            if incluir_prefijo is not None and not img_p.name.startswith(incluir_prefijo):
                continue
            if excluir_prefijo is not None and img_p.name.startswith(excluir_prefijo):
                continue
            imgs_neg.append(img_p)
    if not imgs_neg:
        return None, 0
    n_fp = 0
    for img_p in imgs_neg:
        res = modelo.predict(source=str(img_p), conf=0.25, verbose=False)[0]
        if res.boxes is not None and len(res.boxes) > 0:
            n_fp += 1
    return n_fp / len(imgs_neg), len(imgs_neg)

for nombre_modelo, modelo in [("CON externos", modelo_con_externos), ("SIN externos", modelo_sin_externos)]:
    fp_ahmed, n_ahmed = tasa_fp_dataset_completo(modelo, DATASET_V2, excluir_prefijo="healthy_")
    fp_chaudhary, n_chaudhary = tasa_fp_dataset_completo(modelo, DATASET_V2, incluir_prefijo="healthy_")
    print(f"\n--- {nombre_modelo} (dataset completo, no solo test) ---")
    print(f"  Negativos Ahmed genuinos: n={n_ahmed}  FP%={fp_ahmed:.1%}")
    print(f"  Negativos Chaudhary:      n={n_chaudhary}  FP%={fp_chaudhary:.1%}")



--- CON externos (dataset completo, no solo test) ---
  Negativos Ahmed genuinos: n=36  FP%=30.6%
  Negativos Chaudhary:      n=219  FP%=2.3%

--- SIN externos (dataset completo, no solo test) ---
  Negativos Ahmed genuinos: n=36  FP%=27.8%
  Negativos Chaudhary:      n=219  FP%=16.0%


### Verification — count of negatives by criterion, to reconcile 255 vs. 112


In [ ]:
n_por_txt_vacio = 0
n_por_sin_txt = 0
n_healthy_prefix = 0
n_total_imagenes = 0

for split in ["train", "val", "test"]:
    carpeta_img = DATASET_V2 / "images" / split
    for img_p in carpeta_img.glob("*"):
        n_total_imagenes += 1
        lbl_p = DATASET_V2 / "labels" / split / f"{img_p.stem}.txt"
        if img_p.name.startswith("healthy_"):
            n_healthy_prefix += 1
        if lbl_p.exists() and not lbl_p.read_text().strip():
            n_por_txt_vacio += 1
        if not lbl_p.exists():
            n_por_sin_txt += 1

print(f"Total imágenes en DATASET_V2 (train+val+test): {n_total_imagenes}")
print(f"Negativos por criterio 'label existe y vacío':  {n_por_txt_vacio}")
print(f"Negativos por criterio 'label no existe':       {n_por_sin_txt}")
print(f"Imágenes con prefijo 'healthy_':                {n_healthy_prefix}")


Total imágenes en DATASET_V2 (train+val+test): 1664
Negativos por criterio 'label existe y vacío':  255
Negativos por criterio 'label no existe':       0
Imágenes con prefijo 'healthy_':                219
